# Topical Lectures, Controls
## Real drone, part 3: the position loop in flight

Andreas Freise, 30.10.2026

### Goal

Part 2 measured what the drone reports. Now we measure how it responds, design the position loop from that measurement, and fly it. Three flights, each about twenty seconds in the air:

1. **The response of the velocity loop.** Under the firmware's controllers the drone takes off, hovers, receives a velocity command of 0.2 m/s for 1.5 s and lands. The measured velocity follows the command with a delay; a first order fit gives the response time `tau` of the velocity loop, the one number about the actuator side that the design needs.
2. **The step response with the designed gains.** With `tau` known, the position loop closed around the velocity loop is a second order system. We choose how fast it should be and how damped, and the gains follow. The flight holds the position for 4 s and then steps the target by 0.5 m.
3. **Tuning.** Faster, slower, with and without damping, compared with the prediction of the model.

The loop runs at 50 Hz with a fixed period, so the `deltaT` of the controllers is the true loop period, and every command is limited to `v_max`. `crazyflie.fly` runs the flights: connect, check the flow deck, reset the estimator, arm, log, take off to 0.5 m with the onboard controller, hover 2 s, hand `x` and `y` to our controller, hold, step, land.

### Safety

* Fly only in the marked area. One person at the keyboard, one watching the drone, nobody within arm's length while the motors run.
* The person at the keyboard keeps the cursor in the input box. **Enter** aborts and lands. The stop button of the cell also lands; press it once only, a second interrupt cuts the landing short.
* The loop lands by itself when the drone leaves a 1 m box around the start, when no state arrives any more, when the battery drops below 3.1 V under load, or when the position estimate becomes unreliable (flow sensor blocked or broken).
* Change one thing per run. Each run is saved under its own `RUN_LABEL`; the file names also carry the label of the drone and the time of the flight.
* After a tumble or crash: close the connection, `python crazyflie.py recover --label 2` in a terminal (your label in place of the 2), then `python crazyflie.py flow --label 2` while you move the drone by hand over the desk. With a broken flow sensor the deck is still detected and the height still works, but the horizontal position is gone.
* Full battery for the tuning runs. Below 3.5 V at rest the drone lacks thrust and the runs cannot be compared.


In [ ]:
# Importing packages
import numpy as np
import matplotlib.pyplot as plt
import module
import crazyflie

%matplotlib inline

# The label of your drone as it reads on the sticker, as in drone_01_connect:
# LABEL = 2, or LABEL = 'A' for a letter.
LABEL = 'A'
URI = crazyflie.drone_uri(LABEL)


## What we know from part 2

The cell loads the sensor identification of your drone. Without the file the flights below refuse to start.


In [ ]:
ident = crazyflie.load_drone_id()
crazyflie.print_drone_id(ident)
limits = crazyflie.gain_limits(ident)
print(f"noise limits: |Kp| < {limits['Kp_max']:.1f} 1/s, |Kd| < {limits['Kd_max']:.1f}")


## Test of the abort

Run this once per session. A dummy loop of 10 s starts the same Enter listener as the flight; an input box appears (in VS Code at the top of the window, in JupyterLab below the cell). Press Enter in it and the loop must report the abort. If it does not, do not fly from this program.


In [ ]:
crazyflie.bench_test_abort()


## Status before take-off

Battery above 3.5 V, flow deck attached, not locked. Run this before every flight.


In [ ]:
with crazyflie.connect(URI) as scf:
    crazyflie.status(scf)


## Flight 1: how fast does the drone follow a velocity command?

Put the drone flat on the floor in the flying area, with its forward side (`+x`, between the motors M1 and M4) pointing into the free space. The drone takes off, hovers, and after 1.5 s receives the velocity command 0.2 m/s along `x` for 1.5 s, then 0 again, then lands. It moves about 30 cm. No loop of ours is closed; this is the firmware alone.


In [ ]:
run_id = crazyflie.identify_response(step=0.2, step_s=1.5, uri=URI)
name_id = crazyflie.save(run_id, 'identify')


## The response time of the velocity loop

The measured velocity after the command step is fitted with a first order response with a dead time,

$$v(t) = v_{cmd}\,\bigl(1 - e^{-(t - t_d)/\tau}\bigr) \quad \text{for } t > t_d .$$

`tau` is the time in which the velocity reaches 63 percent of the command; `t_d` is the delay before anything happens (radio, estimator, attitude loop). The fit is a grid search, nothing more. Look at the plot: a first order response has no overshoot. If the measured velocity overshoots or oscillates, the inner loops are not well described by one time constant, and the design below will be optimistic.


In [ ]:
fit = crazyflie.fit_tau(run_id)
tau = fit['tau_s']

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(fit['t'], fit['v_meas'], label='measured vx')
ax.plot(fit['t'], fit['v_fit'], 'k--', label=f"first order fit, tau = {tau:.2f} s, delay {fit['delay_s']:.2f} s")
ax.axhline(fit['v_cmd'], color='0.5', lw=0.8, label='command')
ax.axhline(0.63 * fit['v_cmd'], color='0.8', lw=0.8, ls=':')
ax.set_xlabel('time since the command step [s]')
ax.set_ylabel('velocity [m/s]')
ax.legend(loc='lower right')
ax.grid()


## From the response time to the gains

Our controller commands the velocity $v_{cmd} = K_p\,(x - x_0) + K_d\,\dot x$, with the sign convention of the simulator (error = value minus setpoint, so both gains are negative). The velocity loop of the firmware follows that command with the time constant we just measured, $\tau\,\ddot x = v_{cmd} - \dot x$. Together,

$$\tau\,\ddot x + (1 - K_d)\,\dot x - K_p\,(x - x_0) = 0 ,$$

a second order system with

$$\omega_0 = \sqrt{-K_p/\tau}, \qquad \zeta = \frac{1 - K_d}{2\,\omega_0\,\tau}.$$

The design goes the other way: choose the natural frequency $\omega_0$ (how fast) and the damping $\zeta$ (0.7 gives a fast response with a small overshoot, 1 none), and the gains follow,

$$K_p = -\omega_0^2\,\tau, \qquad K_d = 1 - 2\,\zeta\,\omega_0\,\tau .$$

Two things to notice. The velocity loop damps the position loop by itself, through the term $\dot x / \tau$: with $K_d = 0$ the damping is $\zeta = 1 / (2\,\omega_0\,\tau)$, and for small $\omega_0$ that is already more than enough. $K_d$ comes into play when we want to be fast: a negative $K_d$ adds damping and allows a larger $\omega_0$. If the formula returns a positive $K_d$, the loop would have to remove damping; keep $K_d = 0$ instead and accept the larger $\zeta$. And the model has no limit on the command, while the flight clamps it at `v_max`; for a 0.5 m step the command of a fast loop sits at the limit for the first part of the step, and the response is slower than the model says.

### Task

Run the cell with a few choices of `omega0` and `zeta` and watch the gains and the predicted step. Then compute what the gains of the September flights, `Kp = -1.2` and `Kd = -0.6`, correspond to on your drone.


In [ ]:
omega0 = 2.0     # rad/s, how fast: the time scale of the response is about 1/omega0
zeta = 0.7       # damping: 0.7 a small overshoot, 1.0 none

Kp, Kd = crazyflie.design_gains(tau, omega0=omega0, zeta=zeta)
if Kd > 0:
    print(f"the design asks for Kd = {Kd:+.2f}, which would remove damping; using Kd = 0")
    Kd = 0.0
w0, z = crazyflie.closed_loop_parameters(Kp, Kd, tau)
print(f"tau = {tau:.2f} s: Kp = {Kp:+.2f} 1/s, Kd = {Kd:+.2f}  ->  omega0 = {w0:.2f} rad/s, zeta = {z:.2f}")
print(f"noise limits from part 2: |Kp| < {limits['Kp_max']:.1f}, |Kd| < {limits['Kd_max']:.1f}")

w0_sep, z_sep = crazyflie.closed_loop_parameters(-1.2, -0.6, tau)
print(f"the September gains Kp = -1.2, Kd = -0.6 on this drone: omega0 = {w0_sep:.2f} rad/s, zeta = {z_sep:.2f}")

t_m, x_m, v_m, cmd_m = crazyflie.model_step_response(Kp, Kd, tau, delay=fit['delay_s'])
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_m, x_m, label=f'model, Kp = {Kp:+.2f}, Kd = {Kd:+.2f}')
ax.axhspan(0.5 - 0.15, 0.5 + 0.15, color='0.92', zorder=0)
ax.axhline(0.5, color='k', lw=0.5, ls='--')
ax.set_xlim(-1, 8)
ax.set_xlabel('time since step [s]')
ax.set_ylabel('position from start [m]')
ax.legend(loc='lower right')
ax.grid()


## The controller

The same `PIDController` as in the simulator, one per axis, with the gains from the design. `Kd` acts on the measured velocity, as in `2Dsim_04_velocity_loop`. The command is limited to `v_max = 0.3 m/s`, well below the 1 m/s of the firmware, because the flying area is small. `dt = 0.02 s` is the period of the flight loop and only matters once `Ki` is used.


In [ ]:
dt = 0.02      # s, period of the control loop in flight (50 Hz)
v_max = 0.3    # m/s, velocity command limit, keep it small in a small room

x_ctrl = module.PIDController(Kp=Kp, Ki=0.0, Kd=Kd, deltaT=dt)
y_ctrl = module.PIDController(Kp=Kp, Ki=0.0, Kd=Kd, deltaT=dt)

def position_controller(x, y, vx, vy, x_target, y_target):
    """Position error in, velocity command out, one PID per axis."""
    vx_cmd = x_ctrl(x, vx, x_target)
    vy_cmd = y_ctrl(y, vy, y_target)
    vx_cmd = float(np.clip(vx_cmd, -v_max, v_max))
    vy_cmd = float(np.clip(vy_cmd, -v_max, v_max))
    return vx_cmd, vy_cmd

def reset_controllers():
    """Zero the integrators, called before every flight."""
    x_ctrl.reset()
    y_ctrl.reset()


### Task: check on paper

The drone hovers still 20 cm to the right of its target (`x = 0.2`, `vx = 0`, target `x = 0`). What command does the controller return? Check with the cell below. Then repeat for a drone at the target that moves to the right at 0.2 m/s. In which direction does the command push, and why is that what we want? Part 2 told you that `+x` is forward: which way will the drone move?


In [ ]:
reset_controllers()
print(position_controller(0.2, 0.0, 0.0, 0.0, 0.0, 0.0))
print(position_controller(0.0, 0.0, 0.2, 0.0, 0.0, 0.0))


## Flight 2: hold, then a step of 0.5 m along x

Drone flat on the floor again, forward side into the free space: the step goes in that direction. Announce the flight and run the cell. The printout shows the position at the handover and the step. If the drone drifts away or oscillates visibly: Enter.


In [ ]:
RUN_LABEL = 'w20_z07'            # what this run tries; goes into the file names, change it for every run
AXES = 'x'                       # 'x', 'y' or 'xy', the axes your controller commands
STEP = 0.5                       # m, size of the target step

run = crazyflie.fly(position_controller, axes=AXES, step=STEP, v_max=v_max,
                    reset=reset_controllers, uri=URI)


Save the run right away. The files go to `data/` as `flight_<drone label>_<time>_<RUN_LABEL>_*.csv`, for example `flight_A_1541_w20_z07_pid.csv`, so that files of several drones and flights can share a folder. The rise time, overshoot and settling error are printed.


In [ ]:
name = crazyflie.save(run, RUN_LABEL)


## Measured against predicted

Position along the commanded axis relative to the start, against the time since the step, with the model on top; below, the commanded velocity against the measured one. The grey band is the target within 15 cm, our criterion for settling. The full analysis follows in `drone_04_analysis`.


In [ ]:
r = crazyflie.load_run(name)
if r['t_step'] is None:
    print('this run was aborted before the step, nothing to plot')
else:
    pid = r['pid']
    t = pid['t_s'] - r['t_step']
    t_m, x_m, v_m, cmd_m = crazyflie.model_step_response(Kp, Kd, tau, delay=fit['delay_s'], step=STEP, v_max=v_max)
    fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    for a in r['axes']:
        ax[0].plot(t, pid[a] - pid[f'target_{a}'][0], label=f'{a} position, flight')
        line, = ax[1].plot(t, pid[f'cmd_{a}'], label=f'{a} command')
        ax[1].plot(t, pid[f'v{a}'], color=line.get_color(), alpha=0.5, label=f'{a} measured')
    ax[0].plot(t_m, x_m, 'k--', label='model')
    ax[0].axhspan(STEP - 0.15, STEP + 0.15, color='0.92', zorder=0)
    ax[0].axhline(STEP, color='k', lw=0.5, ls='--')
    ax[0].axhline(0, color='k', lw=0.5, ls='--')
    ax[0].set_ylabel('position from start [m]')
    ax[0].legend(loc='lower right')
    ax[0].grid()
    ax[1].plot(t_m, cmd_m, 'k--', lw=0.8, label='model command')
    ax[1].axhline(v_max, color='k', lw=0.5, ls='--')
    ax[1].axhline(-v_max, color='k', lw=0.5, ls='--')
    ax[1].set_ylabel('velocity [m/s]')
    ax[1].set_xlabel('time since step [s]')
    ax[1].legend(loc='upper right')
    ax[1].grid()
    plt.tight_layout()


## Flight 3 and onwards: tuning

Change one thing per run and keep a table in your notes: label, `omega0`, `zeta`, `Kp`, `Kd`, rise time, overshoot, settling error, and the prediction next to it. Change `omega0` or `zeta` in the design cell, run it and the controller cell, then run the status, flight and save cells again with a new `RUN_LABEL`.

We suggest as experiments:

1. **Faster.** `omega0 = 3.0`, then `4.0`, with `zeta = 0.7`. The gains grow with the square of `omega0`. Does the measured rise time shrink as the model says? At which `omega0` does the command sit at `v_max` for most of the step, so that the loop cannot be faster whatever the gains?
2. **Damping.** `zeta = 0.4` and `zeta = 1.0` at the `omega0` you liked. Where does the overshoot appear in the flight, and where in the model? A difference points at what the first order model leaves out: the dead time and the attitude loop.
3. **Both axes.** `AXES = 'xy'`, the step then goes diagonally. Do the two axes behave the same?
4. **Optional.** A small integrator, `Ki = -0.2` in the controller cell. Where does an integrator help on a drone that drifts with the air in the room, and where does it hurt?

The members of your group who are not at the drone run `drone_04_analysis` meanwhile. It predicts the step response for the same gains, so you can compare prediction and flight before you change the gains.


## If something goes wrong

* `is locked: yes` after a tumble: `python crazyflie.py recover --label 2` in a terminal, then `python crazyflie.py status --label 2`. Still locked: `python crazyflie.py reboot --label 2`, wait a few seconds, connect again.
* `No flow deck detected`: switch off, reseat the deck, switch on.
* The drone wanders off during the hold although the command is small: the floor has too little texture for the flow sensor. Fly over the mat, or check the sensor with `python crazyflie.py flow --label 2`.
* `state samples stale`: the radio link dropped. Check the Crazyradio and the address, keep the laptop close to the flying area.
* The loop never prints `handover`: the take-off failed, usually a low battery or a locked supervisor.
* `No sensor identification found`: run `drone_02_handtest` first, or copy its `data/drone_id.csv` from the laptop that ran it.
